# DrinkScanAI — Build + Convert CoreML Model

This notebook builds the EfficientNet-B0 drink classifier and converts it to CoreML **entirely in Colab** — no file uploads needed.

**Just run all cells top to bottom. At the end, download the model file.**

Total time: ~5 minutes

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────
!pip install coremltools==8.1 timm onnx torch torchvision -q
print('✓ All dependencies installed')

In [ ]:
# ── Cell 2: Define 56-class drink taxonomy ────────────────────────────────
import json

DRINK_CLASSES = {
    'espresso':        {'name': 'Espresso',          'category': 'coffee',       'calories': 9,   'caffeine_mg': 212},
    'americano':       {'name': 'Americano',          'category': 'coffee',       'calories': 3,   'caffeine_mg': 35},
    'coffee_black':    {'name': 'Black Coffee',       'category': 'coffee',       'calories': 1,   'caffeine_mg': 40},
    'latte':           {'name': 'Latte',              'category': 'coffee',       'calories': 54,  'caffeine_mg': 27},
    'cappuccino':      {'name': 'Cappuccino',         'category': 'coffee',       'calories': 40,  'caffeine_mg': 27},
    'flat_white':      {'name': 'Flat White',         'category': 'coffee',       'calories': 60,  'caffeine_mg': 55},
    'mocha':           {'name': 'Mocha',              'category': 'coffee',       'calories': 70,  'caffeine_mg': 27},
    'cold_brew':       {'name': 'Cold Brew Coffee',   'category': 'coffee',       'calories': 5,   'caffeine_mg': 83},
    'iced_coffee':     {'name': 'Iced Coffee',        'category': 'coffee',       'calories': 25,  'caffeine_mg': 30},
    'macchiato':       {'name': 'Macchiato',          'category': 'coffee',       'calories': 13,  'caffeine_mg': 53},
    'black_tea':       {'name': 'Black Tea',          'category': 'tea',          'calories': 1,   'caffeine_mg': 20},
    'green_tea':       {'name': 'Green Tea',          'category': 'tea',          'calories': 1,   'caffeine_mg': 12},
    'matcha_latte':    {'name': 'Matcha Latte',       'category': 'tea',          'calories': 50,  'caffeine_mg': 35},
    'chai_latte':      {'name': 'Chai Latte',         'category': 'tea',          'calories': 62,  'caffeine_mg': 14},
    'herbal_tea':      {'name': 'Herbal Tea',         'category': 'tea',          'calories': 1,   'caffeine_mg': 0},
    'iced_tea':        {'name': 'Iced Tea',           'category': 'tea',          'calories': 16,  'caffeine_mg': 5},
    'bubble_tea':      {'name': 'Bubble Tea',         'category': 'tea',          'calories': 80,  'caffeine_mg': 15},
    'orange_juice':    {'name': 'Orange Juice',       'category': 'juice',        'calories': 45,  'caffeine_mg': 0},
    'apple_juice':     {'name': 'Apple Juice',        'category': 'juice',        'calories': 46,  'caffeine_mg': 0},
    'grape_juice':     {'name': 'Grape Juice',        'category': 'juice',        'calories': 60,  'caffeine_mg': 0},
    'pineapple_juice': {'name': 'Pineapple Juice',    'category': 'juice',        'calories': 50,  'caffeine_mg': 0},
    'mango_juice':     {'name': 'Mango Juice',        'category': 'juice',        'calories': 60,  'caffeine_mg': 0},
    'cranberry_juice': {'name': 'Cranberry Juice',    'category': 'juice',        'calories': 46,  'caffeine_mg': 0},
    'tomato_juice':    {'name': 'Tomato Juice',       'category': 'juice',        'calories': 17,  'caffeine_mg': 0},
    'lemonade':        {'name': 'Lemonade',           'category': 'juice',        'calories': 40,  'caffeine_mg': 0},
    'whole_milk':      {'name': 'Whole Milk',         'category': 'milk',         'calories': 61,  'caffeine_mg': 0},
    'skim_milk':       {'name': 'Skim Milk',          'category': 'milk',         'calories': 34,  'caffeine_mg': 0},
    'oat_milk':        {'name': 'Oat Milk',           'category': 'milk',         'calories': 47,  'caffeine_mg': 0},
    'almond_milk':     {'name': 'Almond Milk',        'category': 'milk',         'calories': 17,  'caffeine_mg': 0},
    'soy_milk':        {'name': 'Soy Milk',           'category': 'milk',         'calories': 54,  'caffeine_mg': 0},
    'coconut_milk':    {'name': 'Coconut Milk Drink', 'category': 'milk',         'calories': 20,  'caffeine_mg': 0},
    'chocolate_milk':  {'name': 'Chocolate Milk',     'category': 'milk',         'calories': 83,  'caffeine_mg': 2},
    'cola':            {'name': 'Cola',               'category': 'soda',         'calories': 42,  'caffeine_mg': 10},
    'diet_cola':       {'name': 'Diet Cola',          'category': 'soda',         'calories': 0,   'caffeine_mg': 12},
    'lemon_lime_soda': {'name': 'Lemon-Lime Soda',    'category': 'soda',         'calories': 42,  'caffeine_mg': 0},
    'ginger_ale':      {'name': 'Ginger Ale',         'category': 'soda',         'calories': 34,  'caffeine_mg': 0},
    'orange_soda':     {'name': 'Orange Soda',        'category': 'soda',         'calories': 48,  'caffeine_mg': 0},
    'root_beer':       {'name': 'Root Beer',          'category': 'soda',         'calories': 41,  'caffeine_mg': 0},
    'water':           {'name': 'Still Water',        'category': 'water',        'calories': 0,   'caffeine_mg': 0},
    'sparkling_water': {'name': 'Sparkling Water',    'category': 'water',        'calories': 0,   'caffeine_mg': 0},
    'coconut_water':   {'name': 'Coconut Water',      'category': 'water',        'calories': 19,  'caffeine_mg': 0},
    'flavored_water':  {'name': 'Flavored Water',     'category': 'water',        'calories': 8,   'caffeine_mg': 0},
    'energy_drink':    {'name': 'Energy Drink',       'category': 'energy_drink', 'calories': 45,  'caffeine_mg': 32},
    'sports_drink':    {'name': 'Sports Drink',       'category': 'sports',       'calories': 25,  'caffeine_mg': 0},
    'protein_shake':   {'name': 'Protein Shake',      'category': 'sports',       'calories': 60,  'caffeine_mg': 0},
    'fruit_smoothie':  {'name': 'Fruit Smoothie',     'category': 'smoothie',     'calories': 62,  'caffeine_mg': 0},
    'green_smoothie':  {'name': 'Green Smoothie',     'category': 'smoothie',     'calories': 40,  'caffeine_mg': 0},
    'hot_chocolate':   {'name': 'Hot Chocolate',      'category': 'hot_drink',    'calories': 67,  'caffeine_mg': 5},
    'milkshake':       {'name': 'Milkshake',          'category': 'hot_drink',    'calories': 112, 'caffeine_mg': 0},
    'beer':            {'name': 'Beer',               'category': 'alcohol',      'calories': 43,  'caffeine_mg': 0},
    'wine_red':        {'name': 'Red Wine',           'category': 'alcohol',      'calories': 85,  'caffeine_mg': 0},
    'wine_white':      {'name': 'White Wine',         'category': 'alcohol',      'calories': 82,  'caffeine_mg': 0},
    'cocktail':        {'name': 'Cocktail',           'category': 'alcohol',      'calories': 100, 'caffeine_mg': 0},
    'kombucha':        {'name': 'Kombucha',           'category': 'fermented',    'calories': 13,  'caffeine_mg': 6},
    'kefir':           {'name': 'Kefir',              'category': 'fermented',    'calories': 40,  'caffeine_mg': 0},
    'unknown':         {'name': 'Unknown Drink',      'category': 'unknown',      'calories': 30,  'caffeine_mg': 0},
}

CLASS_IDS    = list(DRINK_CLASSES.keys())
CLASS_NAMES  = [DRINK_CLASSES[k]['name'] for k in CLASS_IDS]
NUM_CLASSES  = len(CLASS_IDS)
print(f'✓ {NUM_CLASSES} drink classes defined')

In [ ]:
# ── Cell 3: Build model and export to ONNX (inline, no file upload needed) ──
import torch
import torch.nn as nn
import timm
import io

IMG_SIZE = 224

print('Downloading EfficientNet-B0 from HuggingFace timm...')
backbone = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0)
backbone.eval()

with torch.no_grad():
    feat_dim = backbone(torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)).shape[-1]
print(f'  Feature dimension: {feat_dim}')

class DrinkClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone   = backbone
        self.classifier = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, NUM_CLASSES),
        )
        # Xavier init
        for layer in self.classifier:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                nn.init.zeros_(layer.bias)

    def forward(self, x):
        return self.classifier(self.backbone(x))

model = DrinkClassifier()
model.eval()

# Export to ONNX in memory (no external data files)
print('Exporting to ONNX...')
dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)
onnx_buffer = io.BytesIO()

with torch.no_grad():
    torch.onnx.export(
        model, dummy, onnx_buffer,
        opset_version=17,
        input_names=['image'],
        output_names=['logits'],
        do_constant_folding=True,
    )

onnx_buffer.seek(0)
onnx_bytes = onnx_buffer.read()
print(f'  ONNX size: {len(onnx_bytes)/1024/1024:.1f} MB')
print('✓ ONNX export done')

In [ ]:
# ── Cell 4: Convert ONNX → CoreML ────────────────────────────────────────
import coremltools as ct
import onnx
import numpy as np
from PIL import Image

print('Loading ONNX from memory...')
onnx_model = onnx.load_from_string(onnx_bytes)
print(f'  Opset: {onnx_model.opset_import[0].version}')

print('Converting to CoreML (Neural Engine target)...')
mlmodel = ct.convert(
    onnx_model,
    inputs=[ct.ImageType(
        name='image',
        shape=(1, 3, IMG_SIZE, IMG_SIZE),
        scale=1/255.0,
        bias=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
        color_layout=ct.colorlayout.RGB,
    )],
    outputs=[ct.TensorType(name='logits')],
    minimum_deployment_target=ct.target.iOS16,
    compute_units=ct.ComputeUnit.ALL,
)

# Add metadata
mlmodel.short_description = 'DrinkScanAI drink classifier — 56 categories'
mlmodel.author  = 'DrinkScanAI'
mlmodel.version = '1.0'
mlmodel.user_defined_metadata['classes'] = json.dumps([
    {'index': i, 'id': CLASS_IDS[i], 'name': CLASS_NAMES[i]}
    for i in range(NUM_CLASSES)
])
mlmodel.user_defined_metadata['num_classes'] = str(NUM_CLASSES)
mlmodel.user_defined_metadata['input_size']  = str(IMG_SIZE)

OUTPUT_PATH = 'DrinkClassifier.mlpackage'
mlmodel.save(OUTPUT_PATH)
print(f'✓ Saved: {OUTPUT_PATH}')

# Quick validation
print('Validating model...')
dummy_img = Image.fromarray(np.random.randint(0, 255, (IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))
out    = mlmodel.predict({'image': dummy_img})
logits = list(out['logits'][0])
top_i  = logits.index(max(logits))
print(f'  Test prediction: {CLASS_NAMES[top_i]}')
print('✓ Validation passed')

In [ ]:
# ── Cell 5: Download the model ────────────────────────────────────────────
import shutil, os
from google.colab import files

zip_name = 'DrinkClassifier_mlpackage'
shutil.make_archive(zip_name, 'zip', '.', OUTPUT_PATH)
zip_file = zip_name + '.zip'
size_mb  = os.path.getsize(zip_file) / 1024 / 1024
print(f'Downloading {zip_file} ({size_mb:.1f} MB)...')
files.download(zip_file)

print()
print('='*50)
print('DONE! Next steps:')
print('='*50)
print('1. Extract DrinkClassifier_mlpackage.zip')
print('2. Copy DrinkClassifier.mlpackage folder to:')
print('   ios/DrinkScanAI/ML/DrinkClassifier.mlpackage')
print('3. python scripts/patch_xcode_project.py')
print('4. git add . && git commit -m "feat: CoreML" && git push')